In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, mean_absolute_error
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv1D, Conv2D, BatchNormalization, MaxPooling1D, MaxPooling2D,
                                     GlobalAveragePooling1D, GlobalAveragePooling2D, Dense, Reshape, Layer,
                                     Lambda, Add, Multiply, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# ================= 1. 物理参数与环境配置 =================
TYPE_NAMES = ["SingleTone", "Chirp", "Pulse", "HoppingJam", "NoiseFM", 
              "NoiseAM", "Comb", "Mixed", "Satellite"]
FS = 200000 
# 理论真值映射 (带宽, 中心频率)
TRUTH_MAP = {
    0: [500, 0], 1: [60000, 0], 2: [45000, 0], 3: [85000, 0],
    4: [70000, 0], 5: [35000, 0], 6: [65000, 0], 7: [80000, 0], 8: [0, 0]
}

# ================= 2. 特征提取组件 (移除残差结构) =================

def extract_stat_features(x):
    """统计特征分支：均值、标准差、均方根、峭度"""
    mu = tf.reduce_mean(x, axis=1, keepdims=True)
    sigma = tf.math.reduce_std(x, axis=1, keepdims=True)
    rms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=1, keepdims=True))
    kurt = tf.reduce_mean(tf.pow((x - mu) / (sigma + 1e-8), 4), axis=1, keepdims=True) - 3.0
    return tf.concat([mu, sigma, rms, kurt], axis=1)

class PLELayer(Layer):
    """渐进分层专家网络 (PLE)"""
    def __init__(self, num_tasks=3, num_shared=2, num_specific=1, expert_dim=128, **kwargs):
        super().__init__(**kwargs)
        self.num_tasks, self.num_shared, self.num_specific, self.expert_dim = num_tasks, num_shared, num_specific, expert_dim
    def build(self, input_shape):
        self.shared_experts = [Dense(self.expert_dim, activation='relu') for _ in range(self.num_shared)]
        self.specific_experts = [[Dense(self.expert_dim, activation='relu') for _ in range(self.num_specific)] for _ in range(self.num_tasks)]
        self.gates = [Dense(self.num_shared + self.num_specific, activation='softmax') for _ in range(self.num_tasks)]
    def call(self, inputs):
        shared_outs = [ex(inputs) for ex in self.shared_experts]
        task_outputs = []
        for t in range(self.num_tasks):
            spec_outs = [ex(inputs) for ex in self.specific_experts[t]]
            all_experts = tf.stack(shared_outs + spec_outs, axis=1)
            gate_weights = tf.expand_dims(self.gates[t](inputs), axis=-1)
            task_outputs.append(tf.reduce_sum(all_experts * gate_weights, axis=1))
        return task_outputs

# ================= 3. 消融模型组装 (w/o ResNet) =================

def build_mbf_plenet_no_resnet(L=1024, num_classes=9):
    """
    【消融变更点】：
    1. 将 ResNetBlock1D/2D 替换为两层普通的 Conv1D/2D + BN 组合。
    2. 移除 Add() 捷径连接，破坏残差学习机制。
    """
    inputs = Input(shape=(L,), name='input_signal')
    x_t = Reshape((L, 1))(inputs)
    
    # --- 分支 1: 时域普通卷积 ---
    x1 = Conv1D(64, 7, strides=2, padding='same', activation='relu')(x_t)
    x1 = BatchNormalization()(x1)
    x1 = Conv1D(64, 7, padding='same', activation='relu')(x1)
    f_time = GlobalAveragePooling1D()(x1)
    
    # --- 分支 2: 频域普通卷积 ---
    stft = Lambda(lambda x: tf.abs(tf.signal.stft(x, frame_length=128, frame_step=64)))(inputs)
    x_f = Reshape((-1, 65, 1))(stft)
    x2 = Conv2D(64, (3,3), strides=2, padding='same', activation='relu')(x_f)
    x2 = BatchNormalization()(x2)
    x2 = Conv2D(64, (3,3), padding='same', activation='relu')(x2)
    f_freq = GlobalAveragePooling2D()(x2)
    
    # --- 分支 3: 时序深度卷积 (移除残差) ---
    x3 = Conv1D(64, 1, padding='same', activation='relu')(x_t)
    x3 = Conv1D(64, 3, padding='same', groups=64, use_bias=False)(x3)
    f_temporal = GlobalAveragePooling1D()(BatchNormalization()(x3))
    
    # --- 分支 4: 统计特征 ---
    f_stat = Dense(64, activation='relu')(Lambda(extract_stat_features)(inputs))

    # 特征融合
    branches = [f_time, f_freq, f_temporal, f_stat]
    weights = [Dense(1, activation='sigmoid')(b) for b in branches]
    f_fused = Dense(128, activation='relu')(Add()([Multiply()([b, w]) for b, w in zip(branches, weights)]))

    # PLE 任务头
    ple_tasks = PLELayer(num_tasks=3, expert_dim=128)(f_fused)
    det_out = Dense(1, activation='sigmoid', name='det_out')(ple_tasks[0])
    cls_out = Dense(num_classes, activation='softmax', name='cls_out')(Dense(128, activation='relu')(ple_tasks[1]))
    reg_out = Dense(3, activation='linear', name='reg_out')(ple_tasks[2])

    model = Model(inputs, [det_out, cls_out, reg_out])
    model.compile(optimizer=Adam(2e-4),
                  loss={'det_out': 'binary_crossentropy', 'cls_out': 'sparse_categorical_crossentropy', 'reg_out': 'mse'},
                  loss_weights={'det_out': 0.8, 'cls_out': 5.0, 'reg_out': 0.5},
                  metrics={'det_out': 'accuracy', 'cls_out': 'accuracy'})
    return model

# ================= 4. 训练与全维度评估 =================

def load_v5_dataset(path):
    all_x, all_y, all_j = [], [], []
    files = sorted([f for f in os.listdir(path) if f.endswith('_X.npy')])
    for fx in files:
        match = re.search(r'jnr(-?\d+)', fx); jnr = float(match.group(1)) if match else 0.0
        x, y = np.load(os.path.join(path, fx)), np.load(os.path.join(path, fx.replace('_X.npy', '_Y.npy')))
        if x.ndim == 3: x = x[:, :, 0]
        all_x.append(x); all_y.append(y); all_j.append(np.full(len(y), jnr))
    X, Y, J = np.vstack(all_x), np.concatenate(all_y), np.concatenate(all_j)
    params = np.stack([np.array([TRUTH_MAP[l][0]/FS for l in Y]), 
                       np.array([TRUTH_MAP[l][1]/(FS/2) for l in Y]), 
                       np.clip((10**(J/10)-0.1)/(1000-0.1), 0, 1)], axis=1).astype(np.float32)
    return train_test_split(X, (Y<8).astype(float), Y, params, J, test_size=0.3, stratify=Y, random_state=42)

def run_scientific_evaluation(model, X_test, d_true, y_true, p_true):
    preds = model.predict(X_test, batch_size=128)
    det_p = (preds[0] > 0.5).astype(int).ravel()
    cls_p = np.argmax(preds[1], 1)
    reg_p = preds[2]

    print("\n" + "="*20 + " 📊 消融实验结果 (w/o ResNet) " + "="*20)
    print(f"✅ 干扰检测准确率: {accuracy_score(d_true, det_p)*100:.2f}%")
    print(f"🎯 总体识别准确率: {accuracy_score(y_true, cls_p)*100:.2f}%")

    p_names = ['带宽 (BW)', '中心频率 (Fc)', '干扰强度 (JNR)']
    maes = mean_absolute_error(p_true, reg_p, multioutput='raw_values')
    for i in range(3):
        rmse = np.sqrt(np.mean((p_true[:, i] - reg_p[:, i])**2))
        safe_denom = np.max(p_true[:, i]) - np.min(p_true[:, i])
        nrmse = rmse / safe_denom if safe_denom > 1e-4 else rmse
        print(f"   🔹 {p_names[i]:<10}: MAE = {maes[i]:.4f} | NRMSE = {nrmse:.4f}")

def main():
    path = "/root/autodl-tmp/validate/0218/dataset_final_ready_v5"
    X_train, X_tmp, d_train, d_tmp, y_train, y_tmp, p_train, p_tmp, j_train, j_tmp = load_v5_dataset(path)
    X_val, X_test, d_val, d_test, y_val, y_test, p_val, p_test, j_val, j_test = \
        train_test_split(X_tmp, d_tmp, y_tmp, p_tmp, j_tmp, test_size=1/3, stratify=y_tmp, random_state=42)
    
    model = build_mbf_plenet_no_resnet()
    callbacks = [EarlyStopping(monitor='val_cls_out_accuracy', patience=15, restore_best_weights=True, mode='max')]
    
    print("🔥 启动消融实验：移除残差连接训练...")
    model.fit(X_train, {'det_out': d_train, 'cls_out': y_train, 'reg_out': p_train},
              validation_data=(X_val, {'det_out': d_val, 'cls_out': y_val, 'reg_out': p_val}),
              epochs=50, batch_size=128, callbacks=callbacks, verbose=1)

    run_scientific_evaluation(model, X_test, d_test, y_test, p_test)

if __name__ == "__main__": main()

2026-02-20 21:12:30.077890: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-20 21:12:30.141644: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-20 21:12:31.034119: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


2026-02-20 21:12:34.716851: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1635] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22188 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090 D, pci bus id: 0000:39:00.0, compute capability: 8.9


🔥 启动消融实验：移除残差连接训练...
Epoch 1/100


2026-02-20 21:12:40.927040: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:424] Loaded cuDNN version 8600
2026-02-20 21:12:41.209878: I tensorflow/compiler/xla/service/service.cc:169] XLA service 0x7f1ac2fb7170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-20 21:12:41.209902: I tensorflow/compiler/xla/service/service.cc:177]   StreamExecutor device (0): NVIDIA GeForce RTX 4090 D, Compute Capability 8.9
2026-02-20 21:12:41.725705: I ./tensorflow/compiler/jit/device_compiler.h:180] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
2026-02-20 21:12:41.735146: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:637] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-20 21:12:41.880519: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` t

745/745 [==============================] - 42s 38ms/step - loss: 6.3932 - det_out_loss: 0.0281 - cls_out_loss: 1.2717 - reg_out_loss: 0.0240 - det_out_accuracy: 0.9892 - cls_out_accuracy: 0.5478 - val_loss: 5.0281 - val_det_out_loss: 1.5160e-06 - val_cls_out_loss: 1.0041 - val_reg_out_loss: 0.0150 - val_det_out_accuracy: 1.0000 - val_cls_out_accuracy: 0.6633
Epoch 2/100
745/745 [==============================] - 25s 34ms/step - loss: 4.6474 - det_out_loss: 3.7919e-07 - cls_out_loss: 0.9284 - reg_out_loss: 0.0111 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.6758 - val_loss: 4.2417 - val_det_out_loss: 2.0612e-07 - val_cls_out_loss: 0.8475 - val_reg_out_loss: 0.0086 - val_det_out_accuracy: 1.0000 - val_cls_out_accuracy: 0.7033
Epoch 3/100
745/745 [==============================] - 25s 34ms/step - loss: 4.0891 - det_out_loss: 1.3877e-07 - cls_out_loss: 0.8170 - reg_out_loss: 0.0084 - det_out_accuracy: 1.0000 - cls_out_accuracy: 0.7080 - val_loss: 3.8483 - val_det_out_loss: 9.9057e-08 -